In [1]:
import array
import gzip
import os
import struct
import time
import urllib.request
from datetime import datetime
from os import path

import equinox as eqx
import jax.nn as jnn
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import optax
import scienceplots  # noqa: F401
from jaxtyping import Array, Float

from snnax import (
    FeedForwardSNN,
    InputLayer,
    SpikingNeuralNet,
    first_spike_decoding,
    spike_latency_encoding,
)

TRAIN = True
SAVE_IDX = datetime.today().strftime("%Y-%m-%d-%H")
key = jr.PRNGKey(1234)

## Load data

In [2]:
_DATA = "./data/"


def _download(url, filename):
    """Download a url to a file in the JAX data temp directory."""
    if not path.exists(_DATA):
        os.makedirs(_DATA)
    out_file = path.join(_DATA, filename)
    if not path.isfile(out_file):
        urllib.request.urlretrieve(url, out_file)
        print(f"downloaded {url} to {_DATA}")


def _partial_flatten(x):
    """Flatten all but the first dimension of an ndarray."""
    return np.reshape(x, (x.shape[0], -1))


def _one_hot(x, k, dtype=np.float32):
    """Create a one-hot encoding of x of size k."""
    return np.array(x[:, None] == np.arange(k), dtype)


def mnist_raw():
    """Download and parse the raw MNIST dataset."""
    # CVDF mirror of http://yann.lecun.com/exdb/mnist/
    base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"

    def parse_labels(filename):
        with gzip.open(filename, "rb") as fh:
            _ = struct.unpack(">II", fh.read(8))
            return np.array(array.array("B", fh.read()), dtype=np.uint8)

    def parse_images(filename):
        with gzip.open(filename, "rb") as fh:
            _, num_data, rows, cols = struct.unpack(">IIII", fh.read(16))
            return np.array(array.array("B", fh.read()), dtype=np.uint8).reshape(
                num_data, rows, cols
            )

    for filename in [
        "train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz",
    ]:
        _download(base_url + filename, filename)

    train_images = parse_images(path.join(_DATA, "train-images-idx3-ubyte.gz"))
    train_labels = parse_labels(path.join(_DATA, "train-labels-idx1-ubyte.gz"))
    test_images = parse_images(path.join(_DATA, "t10k-images-idx3-ubyte.gz"))
    test_labels = parse_labels(path.join(_DATA, "t10k-labels-idx1-ubyte.gz"))

    return train_images, train_labels, test_images, test_labels


def mnist(permute_train=False):
    """Download, parse and process MNIST data to unit scale and one-hot labels."""
    train_images, train_labels, test_images, test_labels = mnist_raw()

    train_images = _partial_flatten(train_images) / np.float32(255.0)
    test_images = _partial_flatten(test_images) / np.float32(255.0)
    train_labels = _one_hot(train_labels, 10)
    test_labels = _one_hot(test_labels, 10)

    if permute_train:
        perm = np.random.RandomState(0).permutation(train_images.shape[0])
        train_images = train_images[perm]
        train_labels = train_labels[perm]

    return train_images, train_labels, test_images, test_labels

In [3]:
train_dat, train_lab, test_dat, test_lab = mnist(permute_train=False)
train_spike_encodings = spike_latency_encoding(train_dat, 0.05, 0.5)

In [5]:
batch_size = 128
t0 = 0
t1 = 2
v_reset = 1.2
v_th = 1
alpha = 3e-2
beta = 5
tau_s = 1
mu = np.array([6, 5])
max_spikes = 100
dt0 = 3e-2
diffusion = False


# Cap exp function for numerical stability
def intensity_fn(v: Float) -> Float:
    return jnp.exp(beta * (jnp.minimum(v - v_th, 5))) / tau_s


class Model(eqx.Module):
    in_size: int
    hidden_size: int
    out_size: int
    w_input: Array
    w: Array
    snn: SpikingNeuralNet
    input_layer: InputLayer
    v0: Array
    i0: Array

    def __init__(self, in_size, hidden_size, out_size, key):
        input_key, snn_key, v0_key, i0_key = jr.split(key, 4)
        num_neurons = hidden_size + out_size
        self.in_size = in_size
        self.out_size = out_size
        self.hidden_size = hidden_size
        self.input_layer = InputLayer(in_size, hidden_size, input_key)
        self.w_input = self.input_layer.w_input
        self.snn = FeedForwardSNN(
            in_size=hidden_size,
            out_size=out_size,
            width_size=0,
            depth=1,
            intensity_fn=intensity_fn,
            v_reset=v_reset,
            alpha=alpha,
            mu=mu,
            key=snn_key,
            diffusion=diffusion,
            wmin=-0.1,
            wmax=0.3,
        )
        self.w = self.snn.w
        self.v0 = jr.uniform(v0_key, (num_neurons,), minval=0.0, maxval=v_th)
        self.i0 = jr.uniform(i0_key, (num_neurons,), minval=0.0, maxval=v_th)

    @eqx.filter_jit
    def __call__(self, x, key):
        batch_size, dim = x.shape
        assert dim == self.in_size
        input_current = self.input_layer(x)
        v0 = jnp.tile(self.v0, (batch_size, 1))
        i0 = jnp.tile(self.i0, (batch_size, 1))
        sol = self.snn(
            input_current,
            t0,
            t1,
            max_spikes,
            batch_size,
            key=key,
            i0=i0,
            v0=v0,
            dt0=dt0,
        )
        first_spike_times = first_spike_decoding(sol, self.out_size)
        return jnn.softmax(-first_spike_times, axis=1).T

In [6]:
key, model_key = jr.split(key)
model = Model(784, 128, 10, model_key)

In [7]:
@eqx.filter_jit
def accuracy(model, batch, key):
    inputs, labels = batch
    labels_class = jnp.argmax(labels, axis=1)
    predicted_class = jnp.argmax(model(inputs, key), axis=1)
    return jnp.mean(predicted_class == labels_class)


@eqx.filter_value_and_grad
def grad_loss(model, batch, key):
    inputs, labels = batch
    preds = model(inputs, key)
    return -jnp.mean(jnp.sum(preds * labels, axis=1))


@eqx.filter_jit
def make_step(model, optim, opt_state, batch, key):
    loss, grads = grad_loss(model, batch, key)
    updates, opt_state = optim.update(grads, opt_state)
    model = eqx.apply_updates(model, updates)
    return loss, model, opt_state


def train(
    key,
    lr=1e-3,
    num_epochs=10,
    batch_size=128,
    momentum=None,
):
    key, model_key = jr.split(key)
    model = Model(784, 64, 10, model_key)

    num_train = train_dat.shape[0]
    num_complete_batches, leftover = divmod(num_train, batch_size)
    num_batches = num_complete_batches + bool(leftover)

    def data_stream():
        while True:
            perm = jr.permutation(key, num_train)
            for i in range(num_batches):
                batch_idx = perm[i * batch_size : (i + 1) * batch_size]
                yield train_dat[batch_idx], train_lab[batch_idx]

    batches = data_stream()

    optim = optax.rmsprop(lr, momentum=momentum)
    opt_state = optim.init(eqx.filter(model, eqx.is_inexact_array))

    for epoch in range(num_epochs):
        epoch_key = jr.fold_in(key, epoch)
        step_key, pred_key = jr.split(epoch_key)
        start_time = time.time()
        for i_batch in range(num_batches):
            batch_key = jr.fold_in(step_key, i_batch)
            loss, model, opt_state = make_step(model, optim, opt_state, next(batches), batch_key)
        epoch_time = time.time() - start_time

        train_acc = accuracy(model, (train_dat, train_lab), pred_key)
        test_acc = accuracy(model, (test_dat, test_lab), pred_key)
        print(f"Epoch {epoch} in {epoch_time:0.2f} sec")
        print(f"Training set accuracy {train_acc}")
        print(f"Test set accuracy {test_acc}")

    return model

In [8]:
train_dat, train_lab = train_dat[:1000], train_lab[:1000]
test_dat, test_lab = test_dat[:1000], test_lab[:1000]

In [9]:
key, train_key = jr.split(key)
train(train_key)

Epoch 0 in 250.82 sec
Training set accuracy 0.10200000554323196
Test set accuracy 0.10000000149011612
Epoch 1 in 49.84 sec
Training set accuracy 0.0990000069141388
Test set accuracy 0.09800000488758087
Epoch 2 in 50.85 sec
Training set accuracy 0.07700000703334808
Test set accuracy 0.10900000482797623
Epoch 3 in 67.71 sec
Training set accuracy 0.07200000435113907
Test set accuracy 0.08000000566244125
Epoch 4 in 86.39 sec
Training set accuracy 0.08900000154972076
Test set accuracy 0.10700000822544098
Epoch 5 in 90.16 sec
Training set accuracy 0.09700000286102295
Test set accuracy 0.08500000089406967
Epoch 6 in 90.72 sec
Training set accuracy 0.09700000286102295
Test set accuracy 0.08500000089406967
Epoch 7 in 91.15 sec
Training set accuracy 0.09700000286102295
Test set accuracy 0.08500000089406967
Epoch 8 in 89.97 sec
Training set accuracy 0.09700000286102295
Test set accuracy 0.08500000089406967
Epoch 9 in 88.38 sec
Training set accuracy 0.09700000286102295
Test set accuracy 0.08500000

Model(
  in_size=784,
  hidden_size=64,
  out_size=10,
  w_input=f32[784,64],
  w=f32[74,74],
  snn=FeedForwardSNN(
    num_neurons=74,
    w=f32[74,74],
    network=bool[74,74](numpy),
    v_reset=1.2,
    alpha=0.03,
    mu=i32[2],
    drift_vf=<function drift_vf>,
    cond_fn=[
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
      <wrapped function cond_fn>,
    